# God's Eye View from a notebook

The globe's `/api` routes are keyless, CORS-open JSON. This notebook uses `examples/gev.py` (urllib only) plus pandas and matplotlib to pull the same numbers the HUD shows, keep their provenance, and plot them.

Set `GEV_BASE_URL` before starting Jupyter to point at a hosted instance (`https://eye.jcamd.com`); the default is a local `npm run dev` on port 3000.

**What these numbers are, and are not.** Home values and rents are Zillow ZHVI / ZORI model indexes, not sales. Jobs and wages are BLS QCEW county aggregates; cells the BLS withholds for confidentiality are `null`, never filled in. Momentum and affordability are estimates computed by the app, and the payload prints the formula. Nothing here is investment, lending or relocation advice, and nothing identifies a person.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), "examples") if os.path.isdir("examples") else os.getcwd())
import pandas as pd
import matplotlib.pyplot as plt
from gev import BASE_URL, GevError, areas, citations, county_history, gauge_history, indicator_history, indicators, report, screen, to_dataframe

print("API:", BASE_URL)

## 1. Texas counties into pandas

`/api/economy?op=areas&bbox=w,s,e,n` returns a GeoJSON FeatureCollection of counties with QCEW employment and wages and Zillow values and rents joined on under `properties.extra`. The route clamps a box to 18 degrees and snaps it to a 1 degree grid, so this Texas box also catches counties across the state line; we keep `extra.stusab == "TX"`. Geometry comes along; `to_dataframe` keeps only the properties, with nested keys dotted (`extra.home.yoyPct`).

In [ ]:
tx_bbox = "-106.7,25.8,-93.5,36.5"
resp = areas(tx_bbox)
print(resp["source"], "| as of:", resp.get("asOf"), "| polygons:", resp.get("polygons"), "| cache age ms:", resp.get("cacheAge"))
df = to_dataframe(resp["data"]["features"])
df = df[df["extra.stusab"] == "TX"] if "extra.stusab" in df else df
print(len(df), "counties")
df[[c for c in df.columns if c.startswith("extra.")]].head()

The columns: `extra.home.latest` / `extra.home.yoyPct` (Zillow ZHVI, USD and 1-year %), `extra.rent.latest` / `extra.rent.yoyPct` (ZORI), `extra.jobs.emp`, `extra.jobs.avgWeeklyWage` and `extra.jobs.yoy.avgWeeklyWage` (BLS QCEW, newest quarter, yoy %). Withheld QCEW rows show as NaN here because the API sent `null`.

In [ ]:
home_yoy, wage_yoy, name_col = "extra.home.yoyPct", "extra.jobs.yoy.avgWeeklyWage", "extra.name"
plot_df = df[[name_col, home_yoy, wage_yoy]].dropna()
print(len(plot_df), "counties with both values;", len(df) - len(plot_df), "have a withheld or missing cell")

## 2. Home-value yoy vs wage yoy

One point per county. Counties above the diagonal saw home values grow faster than wages over the last year. The two axes are percent changes of different things (a model index vs a survey aggregate over different periods), so the comparison is descriptive, not causal.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(plot_df[wage_yoy], plot_df[home_yoy], s=12, alpha=0.6)
lim = [min(plot_df[wage_yoy].min(), plot_df[home_yoy].min()), max(plot_df[wage_yoy].max(), plot_df[home_yoy].max())]
ax.plot(lim, lim, linestyle="--", linewidth=1, color="grey", label="home yoy = wage yoy")
ax.axhline(0, linewidth=0.5, color="black"); ax.axvline(0, linewidth=0.5, color="black")
for _, r in plot_df.nlargest(5, home_yoy).iterrows():
    ax.annotate(r[name_col], (r[wage_yoy], r[home_yoy]), fontsize=8, xytext=(3, 3), textcoords="offset points")
as_of = resp.get("asOf") or {}
ax.set_xlabel(f"Average weekly wage, yoy % (BLS QCEW {as_of.get('qcew', '')})")
ax.set_ylabel(f"Home value, yoy % (Zillow ZHVI {as_of.get('zhvi', '')})")
ax.set_title("Texas counties: home value growth vs wage growth")
ax.legend()
plt.show()

## 3. Travis County momentum history

`/api/economy/history?op=county&fips=48453` returns `data.series[]`, backfilled series for one county, including `momentum:county:48453`. Momentum is an **estimate**: the payload carries the formula in `provenance.method`, and the citation below repeats it. On a server without this route the call raises `GevError(404)`; the cell says so and moves on.

In [ ]:
travis = None
try:
    travis = county_history("48453")
except GevError as e:
    print("county history not available on this server:", e.status, e.body)

if travis:
    series_list = travis["data"]["series"]
    print("series:", [s["id"] for s in series_list])
    momentum = next((s for s in series_list if s["id"].startswith("momentum:")), None)
    if momentum:
        m = to_dataframe(momentum)
        ax = m.plot(x="time", y="v", figsize=(8, 3.5), legend=False, title=momentum.get("title", "Momentum"))
        ax.set_ylabel(momentum.get("unit", ""))
        ax.axhline(0, linewidth=0.5, color="black")
        plt.show()
        print("method:", momentum.get("provenance", {}).get("method"))

## 4. Run a screen

`/api/screen?kind=county&q=...` filters and sorts every county by its published metrics with a small expression language (`<where> [SORT field [ASC|DESC]] [LIMIT n]`; fields such as `home.yoyPct`, `jobs.yoy.emp`, `momentum`, `priceToRent`; `?kind=county&fields=1` lists them). Rows come back with `values` per field used plus `stats` over every match, `fields[]` with units and any estimate's method, and `provenance[]`. Here: counties where home values rose more than 5 % while employment fell, hottest momentum first.

In [ ]:
try:
    hits = screen("county", "home.yoyPct > 5 AND jobs.yoy.emp < 0 SORT momentum DESC LIMIT 25")
    d = hits["data"]
    print(d["count"], "of", d["total"], "matching counties; fields used:", d["fieldsUsed"])
    rows = [{"id": r["id"], "name": r["name"], **r["values"]} for r in d["rows"]]
    display(to_dataframe(rows))
    print("\n".join(citations(hits)))
except GevError as e:
    print("screen not available on this server:", e.status, e.body)

## 5. Mississippi River at Memphis

The stage of the Mississippi at Memphis (USGS 07032000, parameter 00065, feet) is a named indicator with NWS flood categories and low-water conventions attached. `/api/indicators?op=history&id=mississippi-memphis-stage` returns the series (`data.points`, live upstream merged with the stored points); the thresholds sit on the indicator's meta from `op=latest`. If the indicator route is not deployed, `/api/water?op=history` gives the same 365 days straight from USGS.

In [ ]:
stage, thresholds = None, []
try:
    stage = indicator_history("mississippi-memphis-stage")
    pts = stage["data"]["points"]
    meta = indicators(ids=["mississippi-memphis-stage"])["data"]["items"][0]["meta"]
    thresholds = meta.get("thresholds", [])
    print(meta.get("title"), "-", meta.get("whyItMatters"))
except GevError as e:
    print("indicator route not available:", e.status, "- falling back to the gauge history")
    stage = gauge_history("USGS-07032000", "00065")
    pts = [{"t": int(pd.Timestamp(r["time"]).value // 10**6), "v": r["value"]} for r in stage["data"]]

s = to_dataframe(pts)
ax = s.plot(x="time", y="v", figsize=(9, 3.5), legend=False, title="Mississippi River at Memphis, gage height (ft), USGS 07032000")
for t in thresholds:
    ax.axhline(t["value"], linestyle=":", linewidth=1, label=f"{t['level']}: {t['label']} ({t['op']} {t['value']} ft)")
if thresholds:
    ax.legend(fontsize=8)
ax.set_ylabel("ft")
plt.show()
last = s.dropna(subset=["v"]).iloc[-1]
print("latest:", last["time"], last["v"], "ft")

## 6. Citations

Routes that carry a `provenance` list say, per value, which publisher, series, period and upstream URL it came from, when we fetched it, and whether it is published, an estimate (with the method) or a snapshot of a live feed. `citations()` renders those as plain-text lines (the same format as `citation()` in `lib/provenance/types.ts`); older routes without the field get a one-line fallback built from `source`.

In [ ]:
for label, payload in [("areas", resp), ("county history", travis), ("indicator", stage)]:
    if not payload:
        continue
    print(f"--- {label}")
    for line in citations(payload):
        print(line)
r = report(-97.75, 30.3)
print("--- market report")
print("\n".join(citations(r)))
print("affordability basis:", r["data"]["affordability"]["basis"])
print("permalink on the globe:", r.get("globe"))